# Transferência de Estilo Neural — Gatys vs. Fast Style Transfer

**Disciplina:** Visão Computacional — Trabalho Final
**Imagens:** conteúdo = `ufc.jpg` (campus da UFC) · estilo = `pintura.jpg` (*A Noite Estrelada*, Van Gogh)

Esta prática implementa e compara as **duas famílias clássicas** de transferência de estilo:

| | **Gatys et al. (2016)** | **Fast Style Transfer — Johnson et al. (2016)** |
|---|---|---|
| Ideia | Otimiza **a imagem** iterativamente | Treina **uma rede** que estiliza em 1 passada |
| Velocidade | Lento (minutos por imagem) | Tempo real (milissegundos por imagem) |
| Rede VGG | Usada como **função de perda** e otimizada | Usada só como **rede de perda** (perceptual loss) |
| Flexibilidade | Qualquer estilo, sem treino | Uma rede por estilo (precisa treinar) |

**Referências usadas (PDFs em anexo):**
- L. A. Gatys, A. S. Ecker, M. Bethge. *Image Style Transfer Using Convolutional Neural Networks*. CVPR 2016 (e o preprint *arXiv:1508.06576*).
- J. Johnson, A. Alahi, L. Fei-Fei. *Perceptual Losses for Real-Time Style Transfer and Super-Resolution*. ECCV 2016.

> **Como rodar:** feito para o **Google Colab**. Vá em *Ambiente de execução → Alterar o tipo de ambiente de execução → GPU (T4)*. Depois execute as células em ordem.

---
### Glossário rápido (conceitos usados no código)
- **CNN** — *Convolutional Neural Network*: rede neural para imagens. Camadas rasas captam bordas/cores; camadas profundas captam objetos.
- **VGG-16 / VGG-19** — CNNs pré-treinadas no ImageNet. Aqui NÃO as treinamos: servem de "régua" que sabe enxergar imagens.
- **Tensor** — array com várias dimensões (a estrutura básica do PyTorch). Uma imagem vira um tensor `[batch, canais, altura, largura]`.
- **Matriz de Gram** — correlação entre os canais de uma camada. Captura textura/cor/pinceladas (o "estilo") e descarta a posição (o "conteúdo").
- **Pooling** — operação que encolhe a imagem resumindo cada janelinha. `max` = pega o maior; `avg` = pega a média.


## 0. Configuração do ambiente

Verificamos a GPU e importamos as bibliotecas. No Colab o PyTorch e a torchvision já vêm instalados.

In [ ]:
# (Opcional) garante versoes recentes no Colab - descomente se necessario
# !pip install -q torch torchvision

import torch          # biblioteca principal: tensores + autograd (gradientes automaticos)
import torchvision    # modelos prontos (VGG) e utilidades para imagens

print("PyTorch:", torch.__version__)
print("torchvision:", torchvision.__version__)
print("GPU disponivel:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# 'device' decide onde os tensores serao processados:
# GPU (cuda) e ~100x mais rapida que a CPU para estas redes.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Usando dispositivo:", device)

### 0.1 Carregando as imagens

Faça **upload** de `pintura.jpg` (estilo) e `ufc.jpg` (conteúdo). A célula abaixo abre o seletor de arquivos do Colab; se os arquivos já estiverem no diretório, ela apenas confirma.

In [ ]:
import os

STYLE_PATH   = "pintura.jpg"   # imagem de ESTILO  (a pintura de Van Gogh)
CONTENT_PATH = "ufc.jpg"       # imagem de CONTEUDO (o campus da UFC)

# Detecta se estamos no Google Colab (para usar o seletor de upload)
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# Se algum arquivo estiver faltando, pede o upload
faltando = [p for p in (STYLE_PATH, CONTENT_PATH) if not os.path.exists(p)]
if faltando:
    if IN_COLAB:
        print("Faca upload de:", faltando)
        up = files.upload()   # selecione pintura.jpg e ufc.jpg na janela
    else:
        print("Coloque os arquivos no diretorio:", faltando)

# 'assert' interrompe o notebook com uma mensagem clara se faltar imagem
assert os.path.exists(STYLE_PATH),   f"Nao encontrei {STYLE_PATH}"
assert os.path.exists(CONTENT_PATH), f"Nao encontrei {CONTENT_PATH}"
print("OK! Imagens encontradas.")

In [ ]:
import matplotlib.pyplot as plt   # biblioteca de graficos/visualizacao
from PIL import Image              # abertura e manipulacao de imagens

# Mostra conteudo e estilo lado a lado so para conferir o que carregamos
fig, ax = plt.subplots(1, 2, figsize=(13, 5))
ax[0].imshow(Image.open(CONTENT_PATH)); ax[0].set_title("Conteudo - ufc.jpg"); ax[0].axis("off")
ax[1].imshow(Image.open(STYLE_PATH));   ax[1].set_title("Estilo - pintura.jpg"); ax[1].axis("off")
plt.tight_layout(); plt.show()

---
# Parte 1 — Style Transfer de Gatys (otimização)

## A ideia central do artigo

Gatys et al. observaram que, numa CNN treinada para reconhecimento de objetos (a **VGG-19**), as ativações das camadas separam naturalmente **conteúdo** de **estilo**:

- **Conteúdo** = os *mapas de ativação* de uma camada profunda. Camadas profundas capturam a estrutura/objetos de alto nível, descartando a aparência exata dos pixels.
- **Estilo** = as *correlações entre os canais* de várias camadas, capturadas pela **matriz de Gram** $G_{ij}=\sum_k F_{ik}F_{jk}$. A Gram descarta a disposição espacial e guarda apenas textura, cor e padrões — exatamente o "estilo".

Partimos de uma imagem (a própria foto de conteúdo) e a **otimizamos por gradiente descendente** para minimizar simultaneamente:

$$\mathcal{L}_{total} = \alpha\,\mathcal{L}_{conteúdo} + \beta\,\mathcal{L}_{estilo}$$

onde $\mathcal{L}_{conteúdo}$ é o erro quadrático entre os mapas de ativação (imagem vs. conteúdo) e $\mathcal{L}_{estilo}$ é o erro quadrático entre as matrizes de Gram (imagem vs. estilo). A razão $\alpha/\beta$ controla o quanto preservamos a foto vs. a pintura.

Detalhe fiel ao artigo: trocamos o **max-pooling por average-pooling** na VGG, o que (segundo os autores) melhora o fluxo do gradiente e produz resultados visualmente mais agradáveis.

In [ ]:
import torch.nn as nn              # camadas e funcoes de perda prontas
import torch.nn.functional as F     # versoes 'funcionais' (ex.: mse_loss)
import torchvision.transforms as transforms   # pre-processamento de imagem
import torchvision.models as models           # arquiteturas prontas (VGG)
import copy

# Tamanho de trabalho: maior com GPU (512), menor na CPU (256) para nao travar.
# Quanto maior, mais detalhe -> porem mais lento e mais memoria.
imsize = 512 if torch.cuda.is_available() else 256

# 'loader' = sequencia de transformacoes aplicadas a cada imagem:
loader = transforms.Compose([
    transforms.Resize((imsize, imsize)),   # padroniza o tamanho
    transforms.ToTensor(),                 # PIL -> tensor [0,1], formato C x H x W
])

def carregar_imagem(caminho):
    img = Image.open(caminho).convert("RGB")  # garante 3 canais (R,G,B)
    img = loader(img).unsqueeze(0)            # unsqueeze(0): adiciona a dimensao de BATCH -> [1,3,H,W]
    return img.to(device, torch.float)        # envia o tensor para a GPU/CPU

content_img = carregar_imagem(CONTENT_PATH)
style_img   = carregar_imagem(STYLE_PATH)

# A perda de conteudo compara mapas posicao a posicao -> os dois tensores
# precisam ter exatamente o mesmo tamanho.
assert content_img.size() == style_img.size(), "Conteudo e estilo precisam ter o mesmo tamanho."
print("Tensores (batch, canais, altura, largura):", content_img.shape)

In [ ]:
# ------------------------------------------------------------------
# As perdas sao escritas como "camadas" (nn.Module). Truque do tutorial
# oficial do PyTorch: inserimos essas camadas DENTRO da VGG; quando a
# imagem passa por elas, cada uma calcula e guarda a sua perda.
# ------------------------------------------------------------------

class ContentLoss(nn.Module):
    """Mede o CONTEUDO: erro quadratico medio entre os mapas de ativacao."""
    def __init__(self, target):
        super().__init__()
        # target = ativacao da imagem de conteudo nesta camada.
        # .detach() congela esse alvo (ele nao deve receber gradiente).
        self.target = target.detach()
    def forward(self, x):
        # compara a ativacao atual 'x' com o alvo fixo
        self.loss = F.mse_loss(x, self.target)
        return x   # repassa 'x' adiante sem alterar (camada "transparente")

def gram_matrix(x):
    """Matriz de Gram = correlacoes entre canais (representa o ESTILO)."""
    b, c, h, w = x.size()             # batch, canais, altura, largura
    features = x.view(b * c, h * w)   # achata cada canal num vetor (perde a posicao)
    G = torch.mm(features, features.t())   # produto -> correlacao entre todos os canais
    return G.div(b * c * h * w)       # normaliza pelo numero de elementos (estabilidade)

class StyleLoss(nn.Module):
    """Mede o ESTILO: erro quadratico medio entre as matrizes de Gram."""
    def __init__(self, target_feature):
        super().__init__()
        # alvo = Gram da imagem de estilo nesta camada (tambem congelado)
        self.target = gram_matrix(target_feature).detach()
    def forward(self, x):
        self.loss = F.mse_loss(gram_matrix(x), self.target)
        return x

In [ ]:
# Carrega a VGG-19 JA TREINADA (no ImageNet). Usamos so a parte convolucional
# (.features) e em modo .eval() (sem dropout/atualizacao). Nao vamos treina-la.
weights = models.VGG19_Weights.DEFAULT
cnn = models.vgg19(weights=weights).features.to(device).eval()

# A VGG foi treinada com imagens normalizadas por estas medias/desvios (ImageNet).
# Precisamos normalizar nossas imagens do mesmo jeito, senao a rede "ve" errado.
norm_mean = torch.tensor([0.485, 0.456, 0.406]).to(device)
norm_std  = torch.tensor([0.229, 0.224, 0.225]).to(device)

class Normalization(nn.Module):
    """Primeira camada do modelo: aplica (img - media) / desvio."""
    def __init__(self, mean, std):
        super().__init__()
        self.mean = mean.view(-1, 1, 1)   # formato [3,1,1] p/ alinhar com os canais
        self.std  = std.view(-1, 1, 1)
    def forward(self, img):
        return (img - self.mean) / self.std

### O que é uma "ativação"?

Quando uma imagem entra na VGG, ela passa por uma camada de cada vez. **Ativação** é simplesmente **a saída que uma camada produz** ao processar a imagem.

Cada camada convolucional é um conjunto de "detectores de padrão". Cada detector varre a imagem e responde mais forte onde encontra o que procura (uma borda, uma cor, uma curva). Esse mapa de respostas é a ativação — um **tensor** `[1, canais, altura, largura]`, onde cada **canal** é um detector e cada **posição** diz o quanto ele "acendeu" ali.

- Camada **rasa**: um canal acende nas bordas/cores simples.
- Camada **profunda**: um canal acende em estruturas complexas (telhado, montanha…).

No código abaixo, `model(content_img).detach()` passa a foto da UFC pela rede **até a camada atual** e guarda a ativação dali — é o **alvo de conteúdo** ("como a estrutura deve parecer"). Para o estilo guardamos a **matriz de Gram** dessa ativação. O `.detach()` **congela** esses alvos: eles são referências fixas e não recebem gradiente. Pré-calculamos isso uma única vez; depois, a cada passo, comparamos a imagem em construção contra esses alvos.

In [ ]:
# Quais camadas da VGG usamos (nomes seguindo o artigo de Gatys):
content_layers = ["conv_4"]                                          # conv4_2 -> CONTEUDO
style_layers   = ["conv_1", "conv_2", "conv_3", "conv_4", "conv_5"]  # 5 camadas -> ESTILO

def montar_modelo_e_perdas(cnn, mean, std, style_img, content_img,
                           content_layers, style_layers):
    """Reconstroi a VGG camada a camada, inserindo as perdas nos pontos certos."""
    normalization = Normalization(mean, std).to(device)
    content_losses, style_losses = [], []
    model = nn.Sequential(normalization)   # comeca pela normalizacao

    i = 0  # conta apenas as camadas convolucionais (para nomea-las)
    for layer in cnn.children():
        if isinstance(layer, nn.Conv2d):
            i += 1
            name = f"conv_{i}"
        elif isinstance(layer, nn.ReLU):
            name = f"relu_{i}"
            layer = nn.ReLU(inplace=False)   # inplace=False evita conflito com as perdas
        elif isinstance(layer, nn.MaxPool2d):
            name = f"pool_{i}"
            # >>> DETALHE FIEL AO ARTIGO <<<
            # Troca max-pooling por average-pooling: melhora o fluxo do gradiente
            # e da resultados visualmente mais suaves (afirmado por Gatys et al.).
            layer = nn.AvgPool2d(kernel_size=layer.kernel_size,
                                 stride=layer.stride, padding=layer.padding)
        elif isinstance(layer, nn.BatchNorm2d):
            name = f"bn_{i}"
        else:
            raise RuntimeError(f"Camada desconhecida: {layer.__class__.__name__}")

        model.add_module(name, layer)   # empilha a camada no nosso modelo

        # Se esta camada for ponto de medicao, inserimos a perda logo depois dela.
        if name in content_layers:
            target = model(content_img).detach()   # ativacao do conteudo aqui
            cl = ContentLoss(target); model.add_module(f"content_loss_{i}", cl)
            content_losses.append(cl)
        if name in style_layers:
            target = model(style_img).detach()      # ativacao do estilo aqui
            sl = StyleLoss(target); model.add_module(f"style_loss_{i}", sl)
            style_losses.append(sl)

    # Otimizacao: cortamos o modelo logo apos a ultima perda (o resto e inutil).
    for j in range(len(model) - 1, -1, -1):
        if isinstance(model[j], (ContentLoss, StyleLoss)):
            break
    model = model[:j + 1]
    return model, style_losses, content_losses

model, style_losses, content_losses = montar_modelo_e_perdas(
    cnn, norm_mean, norm_std, style_img, content_img, content_layers, style_layers)
model.requires_grad_(False)   # congela os pesos da VGG: NAO treinamos a rede
print("Modelo de perdas montado.")

### O otimizador L-BFGS e a `closure`

Aqui otimizamos **a própria imagem** (não os pesos da rede). Quem decide *como* ajustar os pixels a cada passo é o **otimizador**.

Usamos o **L-BFGS**, o mesmo do artigo de Gatys. Diferente do SGD/Adam (que usam só a inclinação do terreno), o L-BFGS também estima a **curvatura** a partir dos últimos passos — por isso converge para um bom resultado em **bem menos iterações**. O "L" é de *Limited-memory*: ele guarda só o histórico recente para economizar memória. Como otimizamos uma única imagem (e não milhões de pesos), vale a pena usar esse método mais "esperto".

**Por que a `closure`?** O L-BFGS pode **reavaliar a perda várias vezes dentro de um mesmo passo**. Por isso o PyTorch exige uma função (`closure`) que ele possa chamar quantas vezes precisar — ela recalcula a perda e o gradiente e devolve o valor. Detalhes:

- `run = [0]` é um **contador** numa lista (truque de escopo: a função interna consegue *modificar* o conteúdo de uma lista externa, mas não reatribuir uma variável simples). Serve só para imprimir o progresso.
- `clamp_(0, 1)` "prende" os pixels no intervalo válido `[0, 1]` após cada ajuste, mantendo a imagem válida.
- `loss.backward()` é onde o **autograd** calcula o gradiente da perda **em relação aos pixels** da imagem.

In [ ]:
import torch.optim as optim

def run_style_transfer(model, content_img, style_losses, content_losses,
                       num_steps=300, style_weight=1e6, content_weight=1.0):
    """Otimiza a IMAGEM (nao os pesos) para minimizar conteudo + estilo."""
    # Comecamos da propria foto de conteudo (poderia ser ruido aleatorio tambem).
    input_img = content_img.clone()
    input_img.requires_grad_(True)   # >>> a IMAGEM e a variavel a ser otimizada <<<
    model.eval()

    # L-BFGS: otimizador usado no proprio artigo de Gatys (bom para esta tarefa).
    optimizer = optim.LBFGS([input_img])

    run = [0]   # contador de passos (lista para ser editavel dentro da closure)
    while run[0] <= num_steps:
        def closure():
            # mantem os pixels no intervalo valido [0,1]
            with torch.no_grad():
                input_img.clamp_(0, 1)
            optimizer.zero_grad()        # zera gradientes acumulados
            model(input_img)             # passa a imagem -> cada perda se atualiza
            s = sum(sl.loss for sl in style_losses)    # soma as 5 perdas de estilo
            c = sum(cl.loss for cl in content_losses)  # perda de conteudo
            # perda total ponderada: style_weight/content_weight = razao beta/alfa
            loss = style_weight * s + content_weight * c
            loss.backward()              # autograd: calcula gradiente em relacao a IMAGEM
            run[0] += 1
            if run[0] % 50 == 0:
                print(f"passo {run[0]:4d} | estilo: {style_weight*s.item():.2f} "
                      f"| conteudo: {content_weight*c.item():.4f}")
            return loss
        optimizer.step(closure)          # ajusta os pixels na direcao que reduz a perda

    with torch.no_grad():
        input_img.clamp_(0, 1)           # garante pixels validos no final
    return input_img

Agora executamos a otimização. Com GPU leva ~1–2 min. A razão entre `style_weight` e `content_weight` é o parâmetro $\beta/\alpha$ do artigo: aumente `style_weight` para uma estilização mais forte.

In [ ]:
import time

t0 = time.time()
# style_weight muito > content_weight -> prioriza o estilo da pintura
output_gatys = run_style_transfer(
    model, content_img, style_losses, content_losses,
    num_steps=300, style_weight=1e6, content_weight=1.0)
tempo_gatys = time.time() - t0
print(f"\nGatys concluido em {tempo_gatys:.1f} s")

In [ ]:
def tensor_para_imagem(t):
    """Converte o tensor [1,3,H,W] de volta para uma imagem PIL exibivel."""
    img = t.cpu().clone().squeeze(0).clamp(0, 1)  # tira batch e fixa em [0,1]
    return transforms.ToPILImage()(img)

# Mostra conteudo, estilo e resultado lado a lado
fig, ax = plt.subplots(1, 3, figsize=(18, 6))
ax[0].imshow(tensor_para_imagem(content_img)); ax[0].set_title("Conteudo"); ax[0].axis("off")
ax[1].imshow(tensor_para_imagem(style_img));   ax[1].set_title("Estilo");   ax[1].axis("off")
ax[2].imshow(tensor_para_imagem(output_gatys)); ax[2].set_title("Resultado - Gatys"); ax[2].axis("off")
plt.tight_layout(); plt.show()

tensor_para_imagem(output_gatys).save("resultado_gatys.png")
print("Salvo em resultado_gatys.png")

---
# Parte 2 — Fast Style Transfer (Johnson et al.)

## O problema que o artigo resolve

O método de Gatys é lindo, mas **lento**: cada imagem exige centenas de passos de otimização. Johnson et al. propõem treinar **uma rede feed-forward** (a *Image Transformation Network*) que aprende a estilizar; depois, estilizar é **uma única passada para frente** — milhares de vezes mais rápido.

## Como funciona o treino

Duas redes entram em jogo:

1. **Rede de transformação** $f_W$ (encoder → blocos residuais → decoder): é a rede que **treinamos**. Recebe uma foto e devolve a versão estilizada.
2. **Rede de perda** (VGG-16 pré-treinada, **congelada**): não é treinada; serve só para *medir* a qualidade, calculando as mesmas perdas perceptuais de Gatys.

Para cada imagem de conteúdo $x$ de um grande conjunto (o artigo usa o **MS-COCO**):

- geramos $\hat{y}=f_W(x)$;
- **perda de conteúdo** (*feature reconstruction*): MSE entre as ativações de $\hat{y}$ e de $x$ numa camada da VGG (`relu2_2`);
- **perda de estilo** (*style reconstruction*): MSE entre as matrizes de Gram de $\hat{y}$ e da imagem de estilo, em várias camadas (`relu1_2, relu2_2, relu3_3, relu4_3`);
- ajustamos os pesos $W$ por gradiente descendente.

Note que a perda é **a mesma** de Gatys — a diferença é que aqui ela treina uma rede em vez de otimizar uma única imagem. A imagem de estilo (`pintura.jpg`) fica fixa: a rede aprende **um** estilo.

### 2.1 A rede de transformação

Esta é a rede que **vamos treinar**. Ela é montada com 3 "peças de Lego" personalizadas — entender as peças é entender a rede:

**`ConvLayer` (convolução com *reflection padding*)** — toda convolução encolhe a imagem nas bordas; o preenchimento normal usa **zeros** (cor preta falsa), o que cria molduras/linhas escuras artificiais. O *reflection padding* preenche **espelhando** os próprios pixels da imagem → bordas naturais, **sem artefatos**.

**`InstanceNorm2d` (normalização por imagem)** — normalizar (média ~0, desvio ~1) estabiliza e acelera o treino. A *Instance*Norm normaliza **cada imagem separadamente** (a *Batch*Norm misturaria o lote). Como o "estilo" depende muito do contraste/cores de cada imagem, normalizar imagem a imagem melhora muito o resultado (melhoria de Ulyanov adotada na prática).

**`ResidualBlock` (atalho `x + f(x)`)** — em vez de gerar a saída do zero, o bloco aprende só **o que somar à entrada**. Esse atalho (ideia das ResNets) deixa o gradiente fluir direto (treino estável) e facilita **preservar a estrutura** da foto: se nada precisa mudar, `f(x) ≈ 0` e a imagem passa intacta. São **5 blocos** empilhados — é neles que o estilo é aplicado.

**`UpsampleConvLayer` (ampliar sem "xadrez")** — para voltar ao tamanho original ampliamos por interpolação simples e só depois aplicamos uma convolução. Isso evita o padrão de **tabuleiro de xadrez** que a convolução transposta costuma gerar.

**Formato geral (ampulheta encoder→decoder):** o *encoder* comprime a imagem (256→128→64 px, `stride=2`) aumentando os canais (3→32→64→128) para "enxergar" regiões maiores; os 5 blocos residuais aplicam o estilo; o *decoder* reconstrói a resolução (64→128→256) e volta a 3 canais RGB. A saída **não tem ReLU final**: sai na escala **`[0, 255]`** (por isso o `.mul(255)` aparece nas transformações adiante).

In [ ]:
class ConvLayer(nn.Module):
    """Convolucao com 'reflection padding' (espelha a borda -> menos artefatos)."""
    def __init__(self, in_c, out_c, kernel, stride):
        super().__init__()
        pad = kernel // 2
        self.pad = nn.ReflectionPad2d(pad)
        self.conv = nn.Conv2d(in_c, out_c, kernel, stride)
    def forward(self, x):
        return self.conv(self.pad(x))

class ResidualBlock(nn.Module):
    """Bloco residual: aprende a CORRECAO (x + f(x)) -> treino mais estavel."""
    def __init__(self, ch):
        super().__init__()
        self.conv1 = ConvLayer(ch, ch, 3, 1); self.in1 = nn.InstanceNorm2d(ch, affine=True)
        self.conv2 = ConvLayer(ch, ch, 3, 1); self.in2 = nn.InstanceNorm2d(ch, affine=True)
        self.relu = nn.ReLU()
    def forward(self, x):
        y = self.relu(self.in1(self.conv1(x)))
        y = self.in2(self.conv2(y))
        return x + y                            # >>> conexao residual (atalho) <<<

class UpsampleConvLayer(nn.Module):
    """Aumenta a resolucao por interpolacao + conv.
    Evita o 'checkerboard' (xadrez) que a conv transposta costuma gerar."""
    def __init__(self, in_c, out_c, kernel, stride, upsample):
        super().__init__()
        self.upsample = upsample
        pad = kernel // 2
        self.pad = nn.ReflectionPad2d(pad)
        self.conv = nn.Conv2d(in_c, out_c, kernel, stride)
    def forward(self, x):
        x = F.interpolate(x, scale_factor=self.upsample, mode="nearest")  # amplia
        return self.conv(self.pad(x))

class TransformerNet(nn.Module):
    """A rede que sera TREINADA. Estrutura: encoder -> residuais -> decoder."""
    def __init__(self):
        super().__init__()
        # ----- ENCODER: reduz a resolucao e aumenta os canais (extrai padroes) -----
        self.conv1 = ConvLayer(3, 32, 9, 1);   self.in1 = nn.InstanceNorm2d(32, affine=True)
        self.conv2 = ConvLayer(32, 64, 3, 2);  self.in2 = nn.InstanceNorm2d(64, affine=True)   # stride 2 -> reduz pela metade
        self.conv3 = ConvLayer(64, 128, 3, 2); self.in3 = nn.InstanceNorm2d(128, affine=True)  # reduz de novo
        # ----- MIOLO: 5 blocos residuais aplicam o "estilo" -----
        self.res = nn.Sequential(*[ResidualBlock(128) for _ in range(5)])
        # ----- DECODER: volta a resolucao original -----
        self.up1 = UpsampleConvLayer(128, 64, 3, 1, upsample=2); self.in4 = nn.InstanceNorm2d(64, affine=True)
        self.up2 = UpsampleConvLayer(64, 32, 3, 1, upsample=2);  self.in5 = nn.InstanceNorm2d(32, affine=True)
        self.out = ConvLayer(32, 3, 9, 1)       # volta a 3 canais (RGB)
        self.relu = nn.ReLU()
    def forward(self, x):
        y = self.relu(self.in1(self.conv1(x)))
        y = self.relu(self.in2(self.conv2(y)))
        y = self.relu(self.in3(self.conv3(y)))
        y = self.res(y)
        y = self.relu(self.in4(self.up1(y)))
        y = self.relu(self.in5(self.up2(y)))
        return self.out(y)                      # saida na escala [0,255] (sem ReLU final)

print("TransformerNet definida.")

### 2.2 A rede de perda (VGG-16) e as perdas perceptuais

Pegamos a VGG-16 pronta e a **fatiamos** em 4 pontos para espiar as ativações de cada um. No `forward` ela devolve as 4 de uma vez (num `namedtuple`, só para acessá-las por nome: `.relu2_2` etc.). O `requires_grad = False` **congela** a VGG: ela nunca é treinada, só serve de **"juíza"** que mede a qualidade.

**A intuição da perda perceptual:** em vez de comparar duas imagens **pixel a pixel** (que puniria qualquer pincelada deslocada), comparamos **como a VGG as enxerga**. Isso captura semelhança de *conteúdo* e de *estilo* de um jeito muito mais próximo da percepção humana.

- `relu2_2` (camada intermediária) → mede **conteúdo** (a estrutura).
- as 4 camadas, via **Gram** → medem **estilo** (textura/cor).

A função `gram` aqui é a versão "em lote" (`f.bmm(...)` processa as 4 imagens do batch de uma vez), e `normalize_batch` converte de `[0,255]` para a normalização do ImageNet que a VGG espera, logo antes de entrar nela.

In [ ]:
from collections import namedtuple

class Vgg16(nn.Module):
    """Rede de PERDA (congelada): extrai ativacoes de 4 camadas da VGG-16.
    Nao e treinada; so serve para MEDIR conteudo e estilo."""
    def __init__(self):
        super().__init__()
        vgg = models.vgg16(weights=models.VGG16_Weights.DEFAULT).features
        # Fatiamos a VGG nos pontos de interesse:
        self.slice1 = nn.Sequential(*[vgg[x] for x in range(4)])     # ate relu1_2
        self.slice2 = nn.Sequential(*[vgg[x] for x in range(4, 9)])  # ate relu2_2
        self.slice3 = nn.Sequential(*[vgg[x] for x in range(9, 16)]) # ate relu3_3
        self.slice4 = nn.Sequential(*[vgg[x] for x in range(16, 23)])# ate relu4_3
        for p in self.parameters():
            p.requires_grad = False   # congela: nenhum peso da VGG e atualizado
    def forward(self, x):
        h = self.slice1(x); r1_2 = h
        h = self.slice2(h); r2_2 = h   # <- usada para a perda de CONTEUDO
        h = self.slice3(h); r3_3 = h
        h = self.slice4(h); r4_3 = h
        Out = namedtuple("Out", ["relu1_2", "relu2_2", "relu3_3", "relu4_3"])
        return Out(r1_2, r2_2, r3_3, r4_3)   # devolve as 4 ativacoes

def gram(x):
    """Versao da matriz de Gram que processa um BATCH inteiro de uma vez."""
    b, c, h, w = x.size()
    f = x.view(b, c, h * w)                  # achata o espaco, mantendo o batch
    G = f.bmm(f.transpose(1, 2)) / (c * h * w)  # bmm = multiplicacao matricial em lote
    return G

def normalize_batch(batch):
    """Normalizacao ImageNet para tensores na escala [0,255] (antes de entrar na VGG)."""
    mean = batch.new_tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
    std  = batch.new_tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
    return ((batch / 255.0) - mean) / std

print("Vgg16 e utilitarios prontos.")

### 2.3 Conjunto de treino (MS-COCO)

Fiel ao artigo, treinamos sobre o **MS-COCO**. Para caber no Colab usamos o split de validação `val2017` (~780 MB, 5 000 imagens) — suficiente para a rede generalizar de forma razoável. O download leva poucos minutos.

In [ ]:
import os

# Baixa o MS-COCO (split de validacao) apenas se ainda nao existir.
# '!' executa comandos de terminal direto no Colab.
if not os.path.exists("coco/val2017"):
    !wget -q http://images.cocodataset.org/zips/val2017.zip   # download (~780 MB)
    !unzip -q val2017.zip                                     # descompacta
    !mkdir -p coco && mv val2017 coco/                        # organiza em coco/val2017
    print("Dataset pronto.")
else:
    print("Dataset ja existe.")

print("Imagens:", len(os.listdir("coco/val2017")))

In [ ]:
from torch.utils.data import DataLoader

IMG_SIZE = 256   # imagens de treino 256x256 (padrao do artigo)
BATCH = 4        # quantas imagens processamos por passo (limite de memoria da GPU)

# Transformacoes aplicadas a cada imagem de treino:
train_tf = transforms.Compose([
    transforms.Resize(IMG_SIZE),           # redimensiona o menor lado p/ 256
    transforms.CenterCrop(IMG_SIZE),       # recorta 256x256 no centro
    transforms.ToTensor(),                 # -> tensor [0,1]
    transforms.Lambda(lambda t: t.mul(255)),  # -> escala [0,255] que a rede usa
])

# ImageFolder espera subpastas (uma por "classe"). A pasta "coco" contem
# a subpasta "val2017", entao isso funciona mesmo sem rotulos reais.
train_ds = torchvision.datasets.ImageFolder("coco", transform=train_tf)
# DataLoader entrega as imagens em lotes, embaralhadas, em paralelo (num_workers).
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True, num_workers=2, drop_last=True)
print("Batches por epoca:", len(train_loader))

### 2.4 Treino da rede

Pesos seguindo o artigo/implementações de referência: `content_weight=1e5`, `style_weight=1e10`, Adam com `lr=1e-3`. `MAX_ITERS` é um teto para um treino rápido na aula — **aumente** para resultados melhores (o ideal são ~2 épocas completas).

**Cada iteração executa 6 passos:**
1. `y = transformer(x)` — a rede **estiliza** um lote de fotos do COCO.
2. `vgg(normalize_batch(...))` — passamos a estilizada `y` e a original `x` pela VGG para ver como ela "enxerga" cada uma.
3. **perda de conteúdo** (`relu2_2`): a estilizada deve manter a **estrutura** da original. **Perda de estilo** (Gram nas 4 camadas): a textura da estilizada deve bater com a da pintura.
4. `loss = c_loss + s_loss` — soma ponderada.
5. `loss.backward()` — o gradiente é calculado **em relação aos PESOS da rede**.
6. `optimizer.step()` — ajusta os pesos da `TransformerNet`.

**A diferença crucial em relação a Gatys (a sacada de Johnson):** a perda é **a mesma** (conteúdo via ativação + estilo via Gram). Muda apenas *o que é otimizado* — em Gatys o `backward()`/`step()` ajustava **os pixels de uma imagem**; aqui ajusta **os pesos de uma rede** treinada sobre milhares de fotos. Depois de treinada, ela estiliza **qualquer** foto numa passada.

Dois detalhes: `gram_style` é pré-calculado **uma vez** fora do laço (a pintura não muda); `gs[:fy.size(0)]` apenas recorta as Gram do estilo quando o último lote tem menos de 4 imagens.

In [ ]:
# A imagem de estilo e FIXA durante todo o treino -> pre-calculamos as Gram dela
# uma unica vez (economiza muito processamento).
style_tf = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Lambda(lambda t: t.mul(255)),
])
style = style_tf(Image.open(STYLE_PATH).convert("RGB")).unsqueeze(0).to(device)
style = style.repeat(BATCH, 1, 1, 1)             # replica p/ casar com o tamanho do batch

vgg = Vgg16().to(device).eval()                  # rede de perda congelada
with torch.no_grad():
    feat_style = vgg(normalize_batch(style))     # ativacoes do estilo nas 4 camadas
gram_style = [gram(f) for f in feat_style]       # Gram alvo (representa o estilo)
print("Gram do estilo pre-computadas.")

In [ ]:
transformer = TransformerNet().to(device)        # a rede que vamos TREINAR
optimizer = optim.Adam(transformer.parameters(), lr=1e-3)  # otimizador dos PESOS
mse = nn.MSELoss()                               # erro quadratico medio

# Pesos das perdas (seguindo o artigo). A razao entre eles controla o
# equilibrio: mais estilo vs. mais fidelidade ao conteudo.
CONTENT_W = 1e5
STYLE_W   = 1e10
EPOCHS    = 2
MAX_ITERS = 1500     # teto p/ treino rapido na aula; aumente p/ melhor qualidade

transformer.train()   # modo treino
it = 0
t0 = time.time()
parar = False
for epoch in range(EPOCHS):
    for x, _ in train_loader:                    # x = lote de fotos do COCO
        x = x.to(device)
        optimizer.zero_grad()                    # zera gradientes do passo anterior

        y = transformer(x)                       # 1) a rede estiliza o lote

        # 2) passamos original e estilizada pela VGG p/ comparar "percepcao"
        feat_y = vgg(normalize_batch(y))
        feat_x = vgg(normalize_batch(x))

        # 3a) PERDA DE CONTEUDO: a estilizada deve manter a estrutura (relu2_2)
        c_loss = CONTENT_W * mse(feat_y.relu2_2, feat_x.relu2_2)

        # 3b) PERDA DE ESTILO: as Gram da estilizada devem bater com as do estilo
        s_loss = 0.0
        for fy, gs in zip(feat_y, gram_style):
            s_loss += mse(gram(fy), gs[: fy.size(0)])
        s_loss *= STYLE_W

        loss = c_loss + s_loss                   # 4) perda total
        loss.backward()                          # 5) gradiente em relacao aos PESOS
        optimizer.step()                         # 6) atualiza os pesos da rede

        it += 1
        if it % 50 == 0:
            print(f"it {it:4d} | conteudo {c_loss.item():.0f} | estilo {s_loss.item():.0f} "
                  f"| {time.time()-t0:.0f}s")
        if it >= MAX_ITERS:
            parar = True; break
    if parar:
        break

print(f"\nTreino concluido: {it} iteracoes em {time.time()-t0:.0f}s")
transformer.eval()                               # modo avaliacao
torch.save(transformer.state_dict(), "fast_style_pintura.pth")  # salva os pesos
print("Modelo salvo em fast_style_pintura.pth")

### 2.5 Estilização em tempo real

Agora o ponto-chave do artigo: estilizar é **uma única passada para frente**. Aplicamos a rede treinada à `ufc.jpg` e medimos o tempo.

In [ ]:
def estilizar_rapido(caminho, net):
    """Estiliza uma imagem com UMA passada na rede treinada (sem otimizacao)."""
    tf = transforms.Compose([
        transforms.ToTensor(),
        transforms.Lambda(lambda t: t.mul(255)),
    ])
    img = tf(Image.open(caminho).convert("RGB")).unsqueeze(0).to(device)
    with torch.no_grad():                        # sem gradientes -> mais rapido
        out = net(img).clamp(0, 255)             # 1 forward pass
    out = out.squeeze(0).cpu() / 255.0           # volta para [0,1]
    return transforms.ToPILImage()(out)

# 1a chamada "aquece" a GPU (aloca memoria); nao a cronometramos.
_ = estilizar_rapido(CONTENT_PATH, transformer)

t0 = time.time()
out_fast = estilizar_rapido(CONTENT_PATH, transformer)
tempo_fast = time.time() - t0
print(f"Fast Style Transfer: {tempo_fast*1000:.1f} ms por imagem")

out_fast.save("resultado_fast.png")

In [ ]:
# Resultado do metodo rapido lado a lado com conteudo e estilo
fig, ax = plt.subplots(1, 3, figsize=(18, 6))
ax[0].imshow(Image.open(CONTENT_PATH)); ax[0].set_title("Conteudo"); ax[0].axis("off")
ax[1].imshow(Image.open(STYLE_PATH));   ax[1].set_title("Estilo");   ax[1].axis("off")
ax[2].imshow(out_fast);                 ax[2].set_title("Resultado - Fast Style Transfer"); ax[2].axis("off")
plt.tight_layout(); plt.show()

---
# Parte 3 — Comparação final

Colocamos os dois resultados lado a lado e comparamos o **tempo por imagem** — o trade-off central entre os dois métodos.

In [ ]:
# Painel 2x2: conteudo, estilo, resultado de Gatys e resultado Fast
fig, ax = plt.subplots(2, 2, figsize=(14, 12))
ax[0,0].imshow(Image.open(CONTENT_PATH)); ax[0,0].set_title("Conteudo (ufc.jpg)"); ax[0,0].axis("off")
ax[0,1].imshow(Image.open(STYLE_PATH));   ax[0,1].set_title("Estilo (pintura.jpg)"); ax[0,1].axis("off")
ax[1,0].imshow(tensor_para_imagem(output_gatys)); ax[1,0].set_title("Gatys (otimizacao)"); ax[1,0].axis("off")
ax[1,1].imshow(out_fast);                 ax[1,1].set_title("Fast Style Transfer (rede)"); ax[1,1].axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# Comparacao numerica de velocidade (o trade-off central entre os metodos)
print("="*48)
print("COMPARACAO DE DESEMPENHO")
print("="*48)
print(f"Gatys  (otimizacao, 300 passos): {tempo_gatys:8.2f} s / imagem")
print(f"Fast   (1 passada na rede)     : {tempo_fast*1000:8.2f} ms / imagem")
print(f"Aceleracao aproximada          : {tempo_gatys/tempo_fast:8.0f} x")

## Conclusões

- **Gatys (otimização):** não precisa de treino e aceita qualquer estilo na hora, mas é lento — cada imagem exige centenas de iterações de otimização sobre os pixels.
- **Fast Style Transfer (Johnson):** investe tempo **uma vez** para treinar uma rede por estilo; depois estiliza em tempo real, milhares de vezes mais rápido. A perda perceptual usada no treino é **exatamente a de Gatys** — a inovação está em usá-la para treinar uma rede feed-forward em vez de otimizar uma única imagem.

**Qualidade vs. velocidade** é o trade-off central. O Fast Style Transfer melhora muito com mais iterações de treino (`MAX_ITERS`) e ajuste dos pesos `CONTENT_W` / `STYLE_W`.

### Possíveis extensões
- Variar a razão $\alpha/\beta$ (Gatys) e observar o efeito.
- Treinar a rede rápida por mais tempo / com o COCO completo.
- Implementar o **AdaIN** (Huang & Belongie, também em anexo) para estilo **arbitrário** em tempo real com uma só rede.